<a href="https://colab.research.google.com/github/RayHan0305/CMEECourseWork/blob/main/MalariaGen_data_pre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pilot FST analysis for Anopheles populations

This notebook performs a pilot population genomic analysis using MalariaGEN Ag3 data.

The aims are:

1. Compile a metadata summary table for female *An. coluzzii* and *An. gambiae*.
2. Identify populations with at least 50 female individuals.
3. Select two *An. coluzzii* populations for pilot analysis.
4. Randomly sample 50 individuals from each population.
5. Access genome data for one 1 Mb region on chromosome 3R.
6. Calculate Hudson's FST for all sites, 0-fold sites, and 4-fold sites.
7. Repeat the same analysis for chromosome X.
8. Record the number of sites entering each FST calculation.
9. Prepare position tables with a distance-from-centromere variable.

In [1]:
!pip install -q "pandas==2.2.2" "malariagen_data>=15,<16" scikit-allel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 70.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.8/215.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.9/775.9 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 M

## Import packages and set output folders

The outputs will be saved under `/content/MalariaGen/`.

The main folders are:

- `data`: metadata tables, selected sample IDs, and position tables
- `results`: FST result tables
- `figures`: plots
- `code`: optional scripts or notebooks

In [3]:
import os
import numpy as np
import pandas as pd
import malariagen_data
import allel

base_dir = "/content/MalariaGen"

code_dir = os.path.join(base_dir, "code")
data_dir = os.path.join(base_dir, "data")
figures_dir = os.path.join(base_dir, "figures")
results_dir = os.path.join(base_dir, "results")

for d in [code_dir, data_dir, figures_dir, results_dir]:
    os.makedirs(d, exist_ok=True)

print("Base folder:", base_dir)
print("Data folder:", data_dir)
print("Results folder:", results_dir)
print("Figures folder:", figures_dir)

Base folder: /content/MalariaGen
Data folder: /content/MalariaGen/data
Results folder: /content/MalariaGen/results
Figures folder: /content/MalariaGen/figures


## Connect to MalariaGEN Ag3 and load sample metadata

This step loads the full Ag3 sample metadata table.

In [4]:
ag3 = malariagen_data.Ag3()

df = ag3.sample_metadata()

print("Metadata shape:", df.shape)
print(df.columns.tolist())

df.to_csv(
    os.path.join(data_dir, "ag3_sample_metadata_full.csv"),
    index=False
)

df.head()

Metadata shape: (25230, 58)
['sample_id', 'partner_sample_id', 'contributor', 'country', 'location', 'year', 'month', 'latitude', 'longitude', 'sex_call', 'sample_set', 'release', 'quarter', 'study_id', 'study_url', 'terms_of_use_expiry_date', 'terms_of_use_url', 'unrestricted_use', 'mean_cov', 'median_cov', 'modal_cov', 'mean_cov_2L', 'median_cov_2L', 'mode_cov_2L', 'mean_cov_2R', 'median_cov_2R', 'mode_cov_2R', 'mean_cov_3L', 'median_cov_3L', 'mode_cov_3L', 'mean_cov_3R', 'median_cov_3R', 'mode_cov_3R', 'mean_cov_X', 'median_cov_X', 'mode_cov_X', 'frac_gen_cov', 'divergence', 'contam_pct', 'contam_LLR', 'is_surveillance', 'aim_species_fraction_arab', 'aim_species_fraction_colu', 'aim_species_fraction_colu_no2l', 'aim_species_gambcolu_arabiensis', 'aim_species_gambiae_coluzzii', 'aim_species', 'country_iso', 'admin1_name', 'admin1_iso', 'admin2_name', 'taxon', 'cohort_admin1_year', 'cohort_admin1_month', 'cohort_admin1_quarter', 'cohort_admin2_year', 'cohort_admin2_month', 'cohort_adm

,sample_id,partner_sample_id,contributor,country,location,year,month,latitude,longitude,sex_call,...,admin1_name,admin1_iso,admin2_name,taxon,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,VBS00256-4651STDY7017184,GP97,Tovi Lehmann,Mali,Dallowere,2012,6,13.616,-7.037,F,...,Koulikouro,ML-2,Banamba,coluzzii,ML-2_colu_2012,ML-2_colu_2012_06,ML-2_colu_2012_Q2,ML-2_Banamba_colu_2012,ML-2_Banamba_colu_2012_06,ML-2_Banamba_colu_2012_Q2
1,VBS00257-4651STDY7017185,GP98,Tovi Lehmann,Mali,Dallowere,2012,6,13.616,-7.037,F,...,Koulikouro,ML-2,Banamba,coluzzii,ML-2_colu_2012,ML-2_colu_2012_06,ML-2_colu_2012_Q2,ML-2_Banamba_colu_2012,ML-2_Banamba_colu_2012_06,ML-2_Banamba_colu_2012_Q2
2,VBS00259-4651STDY7017186,GP100,Tovi Lehmann,Mali,Dallowere,2012,6,13.616,-7.037,F,...,Koulikouro,ML-2,Banamba,coluzzii,ML-2_colu_2012,ML-2_colu_2012_06,ML-2_colu_2012_Q2,ML-2_Banamba_colu_2012,ML-2_Banamba_colu_2012_06,ML-2_Banamba_colu_2012_Q2
3,VBS00262-4651STDY7017187,GP103,Tovi Lehmann,Mali,Dallowere,2012,6,13.616,-7.037,F,...,Koulikouro,ML-2,Banamba,coluzzii,ML-2_colu_2012,ML-2_colu_2012_06,ML-2_colu_2012_Q2,ML-2_Banamba_colu_2012,ML-2_Banamba_colu_2012_06,ML-2_Banamba_colu_2012_Q2
4,VBS00277-4651STDY7017189,GP118,Tovi Lehmann,Mali,Dallowere,2012,6,13.616,-7.037,F,...,Koulikouro,ML-2,Banamba,coluzzii,ML-2_colu_2012,ML-2_colu_2012_06,ML-2_colu_2012_Q2,ML-2_Banamba_colu_2012,ML-2_Banamba_colu_2012_06,ML-2_Banamba_colu_2012_Q2


## Filter female *An. coluzzii* and *An. gambiae* samples

Only female individuals are retained, following the project requirement.

In [5]:
meta_female = df[
    (df["taxon"].isin(["coluzzii", "gambiae"])) &
    (df["sex_call"] == "F")
].copy()

print("Female coluzzii/gambiae metadata shape:", meta_female.shape)

meta_female.to_csv(
    os.path.join(data_dir, "metadata_female_coluzzii_gambiae_sample_level.csv"),
    index=False
)

meta_female[
    ["sample_id", "taxon", "country", "location", "year", "month", "contributor", "sex_call"]
].head()

Female coluzzii/gambiae metadata shape: (16945, 58)


,sample_id,taxon,country,location,year,month,contributor,sex_call
0,VBS00256-4651STDY7017184,coluzzii,Mali,Dallowere,2012,6,Tovi Lehmann,F
1,VBS00257-4651STDY7017185,coluzzii,Mali,Dallowere,2012,6,Tovi Lehmann,F
2,VBS00259-4651STDY7017186,coluzzii,Mali,Dallowere,2012,6,Tovi Lehmann,F
3,VBS00262-4651STDY7017187,coluzzii,Mali,Dallowere,2012,6,Tovi Lehmann,F
4,VBS00277-4651STDY7017189,coluzzii,Mali,Dallowere,2012,6,Tovi Lehmann,F


## Compile the metadata summary table

The requested metadata table contains:

- Species
- Country
- Location
- Year
- Month
- Collector
- Number of individuals, females only

In [6]:
metadata_summary = (
    meta_female
    .groupby(
        ["taxon", "country", "location", "year", "month", "contributor"],
        dropna=False
    )
    .size()
    .reset_index(name="Number_of_individuals_females_only")
)

metadata_summary = metadata_summary.rename(columns={
    "taxon": "Species",
    "country": "Country",
    "location": "Location",
    "year": "Year",
    "month": "Month",
    "contributor": "Collector"
})

metadata_summary = metadata_summary.sort_values(
    "Number_of_individuals_females_only",
    ascending=False
)

metadata_summary.to_csv(
    os.path.join(data_dir, "metadata_summary_female_coluzzii_gambiae.csv"),
    index=False
)

print("Metadata summary shape:", metadata_summary.shape)
metadata_summary.head(30)

Metadata summary shape: (1610, 7)


,Species,Country,Location,Year,Month,Collector,Number_of_individuals_females_only
382,coluzzii,Ghana,Ofoase-Ayirebi,2021,7,Lucas Amenga-Etego,261
336,coluzzii,Ghana,Korle-Bu,2018,2,Alexander Egyir-Yawson,239
961,gambiae,Ghana,Obuasi-A,2017,9,Alexander Egyir-Yawson,198
213,coluzzii,"Gambia, The",Sitahuma,2019,10,Alfred Amambua-Ngwa,178
219,coluzzii,"Gambia, The",Wali Kunda,2012,10,Charles Godfray,172
267,coluzzii,Ghana,Bonia-A,2018,4,Lucas Amenga-Etego,133
271,coluzzii,Ghana,Bonia-A,2018,8,Lucas Amenga-Etego,119
273,coluzzii,Ghana,Bonia-B,2019,9,Lucas Amenga-Etego,119
1593,gambiae,Uganda,Nagongera,2012,10,Martin Donnelly,112
939,gambiae,Ghana,Madina-C,2017,11,Alexander Egyir-Yawson,111


## Identify populations with at least 50 female individuals

Only populations with at least 50 female individuals are eligible for the pilot FST analysis.

In [7]:
eligible_populations = metadata_summary[
    metadata_summary["Number_of_individuals_females_only"] >= 50
].copy()

eligible_populations = eligible_populations.sort_values(
    ["Species", "Number_of_individuals_females_only"],
    ascending=[True, False]
)

eligible_populations.to_csv(
    os.path.join(data_dir, "eligible_populations_min50_females.csv"),
    index=False
)

eligible_coluzzii = eligible_populations[
    eligible_populations["Species"] == "coluzzii"
].copy()

eligible_gambiae = eligible_populations[
    eligible_populations["Species"] == "gambiae"
].copy()

eligible_coluzzii.to_csv(
    os.path.join(data_dir, "eligible_coluzzii_populations_min50_females.csv"),
    index=False
)

eligible_gambiae.to_csv(
    os.path.join(data_dir, "eligible_gambiae_populations_min50_females.csv"),
    index=False
)

print("Eligible populations shape:", eligible_populations.shape)
print(eligible_populations["Species"].value_counts())

eligible_coluzzii.head(30)

Eligible populations shape: (68, 7)
Species
coluzzii    42
gambiae     26
Name: count, dtype: int64


,Species,Country,Location,Year,Month,Collector,Number_of_individuals_females_only
382,coluzzii,Ghana,Ofoase-Ayirebi,2021,7,Lucas Amenga-Etego,261
336,coluzzii,Ghana,Korle-Bu,2018,2,Alexander Egyir-Yawson,239
213,coluzzii,"Gambia, The",Sitahuma,2019,10,Alfred Amambua-Ngwa,178
219,coluzzii,"Gambia, The",Wali Kunda,2012,10,Charles Godfray,172
267,coluzzii,Ghana,Bonia-A,2018,4,Lucas Amenga-Etego,133
271,coluzzii,Ghana,Bonia-A,2018,8,Lucas Amenga-Etego,119
273,coluzzii,Ghana,Bonia-B,2019,9,Lucas Amenga-Etego,119
207,coluzzii,"Gambia, The",Palang Fula,2019,8,Alfred Amambua-Ngwa,102
76,coluzzii,Burkina Faso,Nagare,2022,10,Abdoulaye Diabate,100
260,coluzzii,Ghana,Bonia-A,2017,7,Lucas Amenga-Etego,94


## Select two pilot populations

For the pilot analysis, two *An. coluzzii* populations from Guinea are selected.

Both populations have more than 50 female individuals and were sampled in the same country, year, month, and by the same collector.

The two selected populations are:

1. Guinea, Siguiri_Dankakoro, January 2022
2. Guinea, Siguiri_Kinebakoura, January 2022

In [8]:
pop1 = {
    "Species": "coluzzii",
    "Country": "Guinea",
    "Location": "Siguiri_Dankakoro",
    "Year": 2022,
    "Month": 1,
    "Collector": "Alfred Amambua-Ngwa"
}

pop2 = {
    "Species": "coluzzii",
    "Country": "Guinea",
    "Location": "Siguiri_Kinebakoura",
    "Year": 2022,
    "Month": 1,
    "Collector": "Alfred Amambua-Ngwa"
}

def get_population_samples(meta_df, pop):
    sub = meta_df[
        (meta_df["taxon"] == pop["Species"]) &
        (meta_df["country"] == pop["Country"]) &
        (meta_df["location"] == pop["Location"]) &
        (meta_df["year"] == pop["Year"]) &
        (meta_df["month"] == pop["Month"]) &
        (meta_df["contributor"] == pop["Collector"])
    ].copy()
    return sub

pop1_samples_all = get_population_samples(meta_female, pop1)
pop2_samples_all = get_population_samples(meta_female, pop2)

print("Population 1 total samples:", pop1_samples_all.shape[0])
print("Population 2 total samples:", pop2_samples_all.shape[0])

pop1_samples_all.to_csv(
    os.path.join(data_dir, "pop1_Guinea_Siguiri_Dankakoro_all_samples.csv"),
    index=False
)

pop2_samples_all.to_csv(
    os.path.join(data_dir, "pop2_Guinea_Siguiri_Kinebakoura_all_samples.csv"),
    index=False
)

Population 1 total samples: 81
Population 2 total samples: 81


## Randomly select 50 individuals from each population

A fixed random seed is used so that the sample selection is reproducible.

In [9]:
random_seed = 42

pop1_samples_50 = pop1_samples_all.sample(n=50, random_state=random_seed)
pop2_samples_50 = pop2_samples_all.sample(n=50, random_state=random_seed)

pop1_ids = pop1_samples_50["sample_id"].tolist()
pop2_ids = pop2_samples_50["sample_id"].tolist()

pop1_samples_50.to_csv(
    os.path.join(data_dir, "pop1_Guinea_Siguiri_Dankakoro_selected_50_samples.csv"),
    index=False
)

pop2_samples_50.to_csv(
    os.path.join(data_dir, "pop2_Guinea_Siguiri_Kinebakoura_selected_50_samples.csv"),
    index=False
)

print("First 5 pop1 sample IDs:")
print(pop1_ids[:5])

print("First 5 pop2 sample IDs:")
print(pop2_ids[:5])

First 5 pop1 sample IDs:
['VBS82093-7212STDY13775357', 'VBS82056-7212STDY13775320', 'VBS82083-7212STDY13775347', 'VBS82094-7212STDY13775358', 'VBS82079-7212STDY13775343']
First 5 pop2 sample IDs:
['VBS82001-7212STDY13775265', 'VBS81960-7212STDY13775224', 'VBS81992-7212STDY13775256', 'VBS82002-7212STDY13775266', 'VBS81982-7212STDY13775246']


## Prepare ordered sample metadata and sample indices

The sample order is important for FST calculation.

The first 50 samples are from population 1, and the next 50 samples are from population 2.

In [10]:
selected_metadata = pd.concat(
    [pop1_samples_50, pop2_samples_50],
    axis=0
).reset_index(drop=True)

selected_metadata["pilot_population"] = (
    ["pop1_Guinea_Siguiri_Dankakoro"] * 50 +
    ["pop2_Guinea_Siguiri_Kinebakoura"] * 50
)

selected_metadata.to_csv(
    os.path.join(data_dir, "pilot_selected_100_samples_metadata.csv"),
    index=False
)

selected_ids_ordered = selected_metadata["sample_id"].tolist()

sample_id_to_index = pd.Series(df.index.values, index=df["sample_id"]).to_dict()
selected_indices_ordered = [sample_id_to_index[sid] for sid in selected_ids_ordered]

selected_indices_table = pd.DataFrame({
    "sample_id": selected_ids_ordered,
    "sample_index": selected_indices_ordered,
    "pilot_population": selected_metadata["pilot_population"]
})

selected_indices_table.to_csv(
    os.path.join(data_dir, "pilot_selected_100_sample_indices.csv"),
    index=False
)

print("Total selected samples:", len(selected_ids_ordered))
print("Total selected sample indices:", len(selected_indices_ordered))

selected_indices_table.head()

Total selected samples: 100
Total selected sample indices: 100


,sample_id,sample_index,pilot_population
0,VBS82093-7212STDY13775357,18561,pop1_Guinea_Siguiri_Dankakoro
1,VBS82056-7212STDY13775320,18530,pop1_Guinea_Siguiri_Dankakoro
2,VBS82083-7212STDY13775347,18553,pop1_Guinea_Siguiri_Dankakoro
3,VBS82094-7212STDY13775358,18562,pop1_Guinea_Siguiri_Dankakoro
4,VBS82079-7212STDY13775343,18549,pop1_Guinea_Siguiri_Dankakoro


## Define an FST calculation function

Hudson's FST is calculated from allele counts for the two populations.

Only biallelic SNPs with valid numerator and denominator values are retained.

In [12]:
def calculate_hudson_fst(ds, n_pop1=50, n_pop2=50):

    gt = allel.GenotypeDaskArray(ds["call_genotype"].data)

    pop1_idx = list(range(0, n_pop1))
    pop2_idx = list(range(n_pop1, n_pop1 + n_pop2))

    ac1 = gt.take(pop1_idx, axis=1).count_alleles(max_allele=3).compute()
    ac2 = gt.take(pop2_idx, axis=1).count_alleles(max_allele=3).compute()

    ac_total = ac1 + ac2

    is_biallelic = ac_total.is_biallelic_01()

    ac1_bi = ac1[is_biallelic]
    ac2_bi = ac2[is_biallelic]

    num, den = allel.hudson_fst(ac1_bi, ac2_bi)

    valid = np.isfinite(num) & np.isfinite(den) & (den > 0)

    fst = np.nansum(num[valid]) / np.nansum(den[valid])
    n_sites = int(valid.sum())

    return fst, n_sites

## Run FST for one 1 Mb region on chromosome 3R

The first pilot region is:

`3R:1,000,001-2,000,000`

FST is calculated for:

- all available sites
- 0-fold degenerate sites
- 4-fold degenerate sites

The number of sites entering each calculation is also recorded.

In [13]:
region_3R = "3R:1,000,001-2,000,000"

ds_3R = ag3.snp_calls(
    region=region_3R,
    sample_indices=selected_indices_ordered,
    site_mask="gamb_colu",
    chunks="native"
)

print(ds_3R)
print(ds_3R.data_vars)

<xarray.Dataset> Size: 2GB
Dimensions:                             (variants: 891969, alleles: 4,
                                         samples: 100, ploidy: 2)
Coordinates:
    variant_position                    (variants) int32 4MB dask.array<chunksize=(46292,), meta=np.ndarray>
    variant_contig                      (variants) uint8 892kB dask.array<chunksize=(46292,), meta=np.ndarray>
    sample_id                           (samples) <U36 14kB dask.array<chunksize=(100,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele                      (variants, alleles) |S1 4MB dask.array<chunksize=(46292, 4), meta=np.ndarray>
    variant_filter_pass_gamb_colu_arab  (variants) bool 892kB dask.array<chunksize=(182258,), meta=np.ndarray>
    variant_filter_pass_gamb_colu       (variants) bool 892kB dask.array<chunksize=(182258,), meta=np.ndarray>
    variant_filter_pass_arab            (variants) bool 892kB dask.array<c

In [14]:
fst_3R_all, n_sites_3R_all = calculate_hudson_fst(ds_3R)

print("Region:", region_3R)
print("FST:", fst_3R_all)
print("Number of sites entering calculation:", n_sites_3R_all)

Region: 3R:1,000,001-2,000,000
FST: -3.1998413693689894e-05
Number of sites entering calculation: 64809


In [15]:
def run_fst_for_region_and_site_class(region, site_class, label):
    """
    Load SNP calls for a given region and site class, then calculate Hudson's FST.
    """

    ds = ag3.snp_calls(
        region=region,
        sample_indices=selected_indices_ordered,
        site_mask="gamb_colu",
        site_class=site_class,
        chunks="native"
    )

    fst, n_sites = calculate_hudson_fst(ds)

    return {
        "Population_1": "Guinea_Siguiri_Dankakoro_coluzzii_2022_01",
        "Population_2": "Guinea_Siguiri_Kinebakoura_coluzzii_2022_01",
        "Chromosome": region.split(":")[0],
        "Region": region,
        "Site_class": label,
        "FST": fst,
        "Number_of_sites": n_sites
    }


results_3R = []

results_3R.append({
    "Population_1": "Guinea_Siguiri_Dankakoro_coluzzii_2022_01",
    "Population_2": "Guinea_Siguiri_Kinebakoura_coluzzii_2022_01",
    "Chromosome": "3R",
    "Region": region_3R,
    "Site_class": "all_sites",
    "FST": fst_3R_all,
    "Number_of_sites": n_sites_3R_all
})

results_3R.append(
    run_fst_for_region_and_site_class(
        region=region_3R,
        site_class="CDS_DEG_0",
        label="0-fold"
    )
)

results_3R.append(
    run_fst_for_region_and_site_class(
        region=region_3R,
        site_class="CDS_DEG_4",
        label="4-fold"
    )
)

fst_results_3R = pd.DataFrame(results_3R)

fst_results_3R.to_csv(
    os.path.join(results_dir, "fst_results_3R_all_0fold_4fold_pilot.csv"),
    index=False
)

fst_results_3R

Access SNP calls: ⠴ (0:00:19.27)

Locate CDS_DEG_0 sites:   0%|          | 0/6 [00:00<?, ?it/s]

Access SNP calls: ⠸ (0:00:15.10)

Locate CDS_DEG_4 sites:   0%|          | 0/6 [00:00<?, ?it/s]

,Population_1,Population_2,Chromosome,Region,Site_class,FST,Number_of_sites
0,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,3R,"3R:1,000,001-2,000,000",all_sites,-0.000032,64809
1,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,3R,"3R:1,000,001-2,000,000",0-fold,0.000132,1108
2,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,3R,"3R:1,000,001-2,000,000",4-fold,-0.000252,1753


## Run the same FST analysis for chromosome X

The same workflow is repeated for a 1 Mb region on chromosome X.

The pilot region is:

`X:1,000,001-2,000,000`

In [16]:
region_X = "X:1,000,001-2,000,000"

ds_X = ag3.snp_calls(
    region=region_X,
    sample_indices=selected_indices_ordered,
    site_mask="gamb_colu",
    chunks="native"
)

print(ds_X)
print(ds_X.data_vars)

<xarray.Dataset> Size: 1GB
Dimensions:                             (variants: 761769, alleles: 4,
                                         samples: 100, ploidy: 2)
Coordinates:
    variant_position                    (variants) int32 3MB dask.array<chunksize=(46153,), meta=np.ndarray>
    variant_contig                      (variants) uint8 762kB dask.array<chunksize=(46153,), meta=np.ndarray>
    sample_id                           (samples) <U36 14kB dask.array<chunksize=(100,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele                      (variants, alleles) |S1 3MB dask.array<chunksize=(46153, 4), meta=np.ndarray>
    variant_filter_pass_gamb_colu_arab  (variants) bool 762kB dask.array<chunksize=(162732,), meta=np.ndarray>
    variant_filter_pass_gamb_colu       (variants) bool 762kB dask.array<chunksize=(162732,), meta=np.ndarray>
    variant_filter_pass_arab            (variants) bool 762kB dask.array<c

In [17]:
fst_X_all, n_sites_X_all = calculate_hudson_fst(ds_X)

print("Region:", region_X)
print("FST:", fst_X_all)
print("Number of sites entering calculation:", n_sites_X_all)

Region: X:1,000,001-2,000,000
FST: 0.00010602031928848509
Number of sites entering calculation: 47260


In [18]:
results_X = []

results_X.append({
    "Population_1": "Guinea_Siguiri_Dankakoro_coluzzii_2022_01",
    "Population_2": "Guinea_Siguiri_Kinebakoura_coluzzii_2022_01",
    "Chromosome": "X",
    "Region": region_X,
    "Site_class": "all_sites",
    "FST": fst_X_all,
    "Number_of_sites": n_sites_X_all
})

results_X.append(
    run_fst_for_region_and_site_class(
        region=region_X,
        site_class="CDS_DEG_0",
        label="0-fold"
    )
)

results_X.append(
    run_fst_for_region_and_site_class(
        region=region_X,
        site_class="CDS_DEG_4",
        label="4-fold"
    )
)

fst_results_X = pd.DataFrame(results_X)

fst_results_X.to_csv(
    os.path.join(results_dir, "fst_results_X_all_0fold_4fold_pilot.csv"),
    index=False
)

fst_results_X

Access SNP calls: ⠧ (0:00:12.52)

Locate CDS_DEG_0 sites:   0%|          | 0/7 [00:00<?, ?it/s]

Access SNP calls: ⠴ (0:00:09.09)

Locate CDS_DEG_4 sites:   0%|          | 0/7 [00:00<?, ?it/s]

,Population_1,Population_2,Chromosome,Region,Site_class,FST,Number_of_sites
0,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,X,"X:1,000,001-2,000,000",all_sites,0.000106,47260
1,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,X,"X:1,000,001-2,000,000",0-fold,0.000450,505
2,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,X,"X:1,000,001-2,000,000",4-fold,0.001501,816


## Combine 3R and X results

The 3R and X chromosome results are combined into one pilot result table.

In [19]:
fst_results_3R_X = pd.concat(
    [fst_results_3R, fst_results_X],
    ignore_index=True
)

fst_results_3R_X.to_csv(
    os.path.join(results_dir, "fst_results_3R_X_all_0fold_4fold_pilot.csv"),
    index=False
)

fst_results_3R_X

,Population_1,Population_2,Chromosome,Region,Site_class,FST,Number_of_sites
0,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,3R,"3R:1,000,001-2,000,000",all_sites,-0.000032,64809
1,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,3R,"3R:1,000,001-2,000,000",0-fold,0.000132,1108
2,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,3R,"3R:1,000,001-2,000,000",4-fold,-0.000252,1753
3,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,X,"X:1,000,001-2,000,000",all_sites,0.000106,47260
4,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,X,"X:1,000,001-2,000,000",0-fold,0.000450,505
5,Guinea_Siguiri_Dankakoro_coluzzii_2022_01,Guinea_Siguiri_Kinebakoura_coluzzii_2022_01,X,"X:1,000,001-2,000,000",4-fold,0.001501,816


## Prepare distance-from-centromere tables

A new variable called `Distance_from_Centromere`.

This variable is placed next to the genomic position column.

The exact centromere coordinates for 3R and X is confirmed before final analysis.  
For now, placeholder values are used and should be replaced later.

In [20]:
# Placeholder centromere coordinates.
# These must be replaced with confirmed coordinates before final analysis.

centromere_3R = 1_000_000
centromere_X = 1_000_000

def make_position_table(ds, chromosome, centromere_position):
    positions = ds["variant_position"].values

    table = pd.DataFrame({
        "Chromosome": chromosome,
        "Position": positions
    })

    table["Distance_from_Centromere"] = table["Position"] - centromere_position

    table = table[
        table["Distance_from_Centromere"] >= 1
    ].copy()

    return table

positions_table_3R = make_position_table(
    ds=ds_3R,
    chromosome="3R",
    centromere_position=centromere_3R
)

positions_table_X = make_position_table(
    ds=ds_X,
    chromosome="X",
    centromere_position=centromere_X
)

positions_table_3R.to_csv(
    os.path.join(data_dir, "positions_3R_with_distance_from_centromere.csv"),
    index=False
)

positions_table_X.to_csv(
    os.path.join(data_dir, "positions_X_with_distance_from_centromere.csv"),
    index=False
)

positions_table_3R.head()

,Chromosome,Position,Distance_from_Centromere
0,3R,1000001,1
1,3R,1000002,2
2,3R,1000003,3
3,3R,1000004,4
4,3R,1000005,5


## Interpretation of the pilot results

The current pilot analysis compares two *An. coluzzii* populations from Guinea.

The main expectations are:

- 4-fold sites are expected to behave more like a neutral baseline.
- 0-fold sites are more likely to be affected by selection because mutations at these sites change amino acids.
- If selection contributes to population differentiation, 0-fold sites may show higher FST than 4-fold sites.
- If FST values are close to zero, this suggests little differentiation between the two selected populations in the analysed region.

The chromosome X results can be compared with chromosome 3R to explore whether differentiation differs between sex chromosomes and autosomes.

## Download all outputs

All output files are compressed into one zip file for download.

After downloading, the zip file can be extracted into the local project folder:

`E:\MalariaGen`

In [21]:
import shutil
from google.colab import files

zip_path = "/content/MalariaGen_outputs"

shutil.make_archive(zip_path, "zip", base_dir)

files.download(zip_path + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>